# Diarización por lote, limpieza y anchors de alta confianza

Notebook para:

- procesar todos los `.wav` de una carpeta
- diarizar con **2 hablantes**
- usar **exclusive speaker diarization** si está disponible
- detectar **zonas solapadas** a partir de la diarización regular
- escoger **anchors** por hablante para la siguiente fase de identificación agente/cliente


## FASE 01.0 - Configuración de entorno y rutas


In [1]:
# INSTALACION DE REQUISITOS PREVIOS

from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")

Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS

import sys

import numpy as np
import pandas as pd
import torch

from google.cloud import storage
from IPython.display import clear_output
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.speaker_verification import (
    PretrainedSpeakerEmbedding,
)

# Permitir imports desde src/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Configuración centralizada
from src.config import (
    HF_TOKEN,
    GCS_CLEAN_AUDIO_PREFIX,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    INPUT_DIR,
    OUTPUT_DIR,
    FINAL_RELABEL_DIR,
    DIARIZATION_SUMMARY_CSV,
    DIARIZATION_ALL_REGULAR_SEGMENTS_CSV,
    DIARIZATION_ALL_SCORED_SEGMENTS_CSV,
    DIARIZATION_ALL_VALID_SEGMENTS_CSV,
    DIARIZATION_ALL_ANCHOR_SEGMENTS_CSV,
    DIARIZATION_ERRORS_CSV,
    EMBEDDING_MODEL_ID,
    EMBEDDING_SAMPLE_RATE,
    RELABEL_MIN_MARGIN,
    RELABELING_SUMMARY_BY_AUDIO_CSV,
    ensure_phase01_directories,
    ensure_relabel_directories,
)

# Utilidades genéricas de CSV
from src.io_utils import (
    read_csv_robust,
    write_csv_atomic,
)

# Transporte genérico de archivos hacia/desde GCS
from src.storage_io import (
    download_prefix_to_local,
    download_directory,
    upload_directory,
    upload_file,
)

# Funciones de diarización
from src.diarizacion import (
    DIARIZATION_SEGMENT_COLUMNS,
    SCORED_SEGMENT_COLUMNS,
    ANCHOR_SEGMENT_COLUMNS,
    diarize_audio,
    save_diarization_outputs,
    get_audio_output_paths,
    required_outputs_exist,
    remove_bad_audio_outputs,
    try_restore_audio_outputs_from_gcs,
    upload_audio_outputs_to_gcs,
    rebuild_consolidated_outputs,
)

from src.reetiquetado_embeddings import (
    run_relabeling_pipeline,
    build_validation_table,
    validation_inputs_exist,
)

print("Configuración y funciones importadas correctamente.")

Configuración y funciones importadas correctamente.


In [3]:
# CONFIGURACIÓN GENERAL

if not HF_TOKEN:
    raise ValueError(
        "No se encontró HF_TOKEN. Revisa el archivo .env en la raíz del repositorio."
    )

ensure_phase01_directories()
ensure_relabel_directories()

# Control de ejecución
RE_DIARIZAR = False                  # False = reanuda/salta audios ya procesados
RE_RELABEL = False                   # False = reutiliza el reetiquetado consolidado existente
BATCH_SAVE_EVERY = 25               # guarda consolidado cada 25 audios nuevos
RESUME_FROM_GCS = True               # recupera outputs de diarización desde GCS
UPLOAD_EACH_AUDIO_TO_GCS = True      # sube outputs por audio al terminar

# Parámetros de diarización
NUM_SPEAKERS = 2
USE_EXCLUSIVE_DIARIZATION = True

# Filtros de segmentos
MIN_SEGMENT_DURATION_SEC = 0.70
MIN_RMS_DBFS = -40.0
MAX_GAP_MERGE_SEC = 0.50

# Selección de anchors
MIN_ANCHOR_DURATION_SEC = 1.20
MAX_OVERLAP_RATIO_FOR_ANCHOR = 0.00
INITIAL_EXCLUDE_SEC_FOR_ANCHORS = 1.50
ANCHORS_PER_SPEAKER = 3

print("GCS_CLEAN_AUDIO_PREFIX:", GCS_CLEAN_AUDIO_PREFIX)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("GCS_DIARIZATION_OUTPUT_PREFIX:", GCS_DIARIZATION_OUTPUT_PREFIX)
print("RE_DIARIZAR:", RE_DIARIZAR)
print("RE_RELABEL:", RE_RELABEL)
print("BATCH_SAVE_EVERY:", BATCH_SAVE_EVERY)

GCS_CLEAN_AUDIO_PREFIX: gs://catedras_audio_detection/pipelineA/procesados_UNAV/clean_audios/
INPUT_DIR: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_input_clean_audios
OUTPUT_DIR: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs
GCS_DIARIZATION_OUTPUT_PREFIX: gs://catedras_audio_detection/pipelineA/procesados_UNAV/diarization_outputs/
RE_DIARIZAR: False
RE_RELABEL: False
BATCH_SAVE_EVERY: 25


In [4]:
# CARGA DEL PIPELINE DEL MODELO PYANNOTE.AUDIO PARA LA DIARIZACION 

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

print("pipeline cargada")


pipeline cargada


In [5]:
# CONEXIÓN AUXILIAR A GOOGLE CLOUD STORAGE

gcs_client = storage.Client()

print("Cliente de GCS configurado correctamente.")

Cliente de GCS configurado correctamente.


In [6]:
# DESCARGA LOCAL DE AUDIOS LIMPIOS DESDE GCS

download_result = download_prefix_to_local(
    source_uri=GCS_CLEAN_AUDIO_PREFIX,
    local_dir=INPUT_DIR,
    gcs_client=gcs_client,
    suffix=".wav",
    force=RE_DIARIZAR,
    clear_output_fn=clear_output,
)

if download_result["found"] == 0:
    raise FileNotFoundError(
        f"No se encontraron WAVs en {GCS_CLEAN_AUDIO_PREFIX}. "
        "Revisa que el notebook de limpieza haya subido los "
        "audios limpios a ese prefijo."
    )

print(f"Audios limpios disponibles localmente en: {INPUT_DIR}")

Descarga completada desde: gs://catedras_audio_detection/pipelineA/procesados_UNAV/clean_audios/
Archivos encontrados: 1200
Archivos descargados: 0
Archivos ya vigentes localmente: 1200
Destino local: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_input_clean_audios
Audios limpios disponibles localmente en: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_input_clean_audios


In [7]:
# LECTURA AUTOMATICA DE TODOS LOS ARCHIVOS .WAV

wav_files = sorted(INPUT_DIR.glob("*.wav"))

print(f"Se encontraron {len(wav_files)} archivos WAV")

if not wav_files:
    raise FileNotFoundError(f"No se encontraron archivos .wav en: {INPUT_DIR}")


Se encontraron 1200 archivos WAV


## FASE 01.1 - DIARIZACION DEL CORPUS COMPLETO 

In [8]:
# DIARIZACIÓN ROBUSTA POR LOTES CON CHECKPOINTS

processed_now = 0
skipped_existing = 0
failed_rows = []
total_audios = len(wav_files)

print("Inicio de diarización robusta")
print(f"Total audios: {total_audios}")
print(f"RE_DIARIZAR: {RE_DIARIZAR}")
print(f"BATCH_SAVE_EVERY: {BATCH_SAVE_EVERY}")
print(f"Destino GCS: {GCS_DIARIZATION_OUTPUT_PREFIX}")


for i, audio_path in enumerate(wav_files, start=1):
    paths = get_audio_output_paths(audio_path)

    if (not RE_DIARIZAR) and RESUME_FROM_GCS:
        try_restore_audio_outputs_from_gcs(
            paths,
            gcs_client,
        )

    if (not RE_DIARIZAR) and required_outputs_exist(paths):
        skipped_existing += 1

        clear_output(wait=True)

        print(
            f"[{i}/{total_audios}] "
            f"Saltando ya procesado: {audio_path.name}"
        )

        print(
            f"Procesados nuevos: {processed_now} | "
            f"Reusados: {skipped_existing} | "
            f"Errores: {len(failed_rows)}"
        )

        continue

    # Eliminar únicamente restos de archivos de cero bytes.
    remove_bad_audio_outputs(paths)

    clear_output(wait=True)

    print(
        f"[{i}/{total_audios}] "
        f"Diarizando: {audio_path.name}"
    )

    print(
        f"Procesados nuevos: {processed_now} | "
        f"Reusados: {skipped_existing} | "
        f"Errores: {len(failed_rows)}"
    )

    try:
        result = diarize_audio(
            audio_path,
            pipeline,
            num_speakers=NUM_SPEAKERS,
            use_exclusive_diarization=USE_EXCLUSIVE_DIARIZATION,
            max_gap_merge_sec=MAX_GAP_MERGE_SEC,
            min_segment_duration_sec=MIN_SEGMENT_DURATION_SEC,
            min_rms_dbfs=MIN_RMS_DBFS,
            min_anchor_duration_sec=MIN_ANCHOR_DURATION_SEC,
            max_overlap_ratio_for_anchor=MAX_OVERLAP_RATIO_FOR_ANCHOR,
            initial_exclude_sec_for_anchors=INITIAL_EXCLUDE_SEC_FOR_ANCHORS,
            anchors_per_speaker=ANCHORS_PER_SPEAKER,
        )

        (
            raw_csv_path,
            clean_csv_path,
            anchors_csv_path,
            rttm_path,
        ) = save_diarization_outputs(
            audio_path,
            result,
            OUTPUT_DIR,
        )

        # Diarización regular necesaria para reconstruir el overlap.
        write_csv_atomic(
            result["df_regular"],
            paths["regular"],
            columns=DIARIZATION_SEGMENT_COLUMNS,
        )

        # Rutas reales generadas por save_diarization_outputs.
        paths["scored"] = raw_csv_path
        paths["valid"] = clean_csv_path
        paths["anchors"] = anchors_csv_path
        paths["rttm"] = rttm_path

        if not required_outputs_exist(paths):
            raise RuntimeError(
                "El audio terminó, pero alguno de sus "
                "outputs quedó incompleto o ilegible."
            )

        if UPLOAD_EACH_AUDIO_TO_GCS:
            upload_audio_outputs_to_gcs(
                paths,
                gcs_client,
            )

        processed_now += 1

    except Exception as error:
        failed_rows.append(
            {
                "audio_file": audio_path.name,
                "audio_stem": audio_path.stem,
                "error": str(error),
            }
        )

        continue

    # Checkpoint cada N audios nuevos o al llegar al último audio.
    checkpoint_due = (
        processed_now > 0
        and processed_now % BATCH_SAVE_EVERY == 0
    )

    final_audio = i == total_audios

    if checkpoint_due or final_audio:
        clear_output(wait=True)

        print(
            "Checkpoint: reconstruyendo consolidados "
            f"tras {processed_now} audios nuevos..."
        )

        (
            df_summary,
            df_all_regular_segments,
            df_all_scored_segments,
            df_all_valid_segments,
            df_all_anchor_segments,
        ) = rebuild_consolidated_outputs(
            wav_files,
            gcs_client,
        )

        print(
            "Checkpoint guardado localmente "
            "y subido a GCS."
        )

        print(
            f"Audios consolidados: {len(df_summary)}"
        )


# Checkpoint final aunque el último lote no sea múltiplo exacto.
(
    df_summary,
    df_all_regular_segments,
    df_all_scored_segments,
    df_all_valid_segments,
    df_all_anchor_segments,
) = rebuild_consolidated_outputs(
    wav_files,
    gcs_client,
)


if failed_rows:
    df_diarization_errors = pd.DataFrame(
        failed_rows
    )

    write_csv_atomic(
        df_diarization_errors,
        DIARIZATION_ERRORS_CSV,
    )

    upload_file(
        DIARIZATION_ERRORS_CSV,
        gcs_client,
        gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
        base_dir=OUTPUT_DIR,
    )

else:
    df_diarization_errors = pd.DataFrame(
        columns=[
            "audio_file",
            "audio_stem",
            "error",
        ]
    )

print("\nDiarización finalizada o reanudada correctamente.")
print(f"Audios nuevos procesados en esta ejecución: {processed_now}")
print(f"Audios reutilizados desde outputs previos: {skipped_existing}")
print(f"Errores: {len(failed_rows)}")
print("Resumen:", DIARIZATION_SUMMARY_CSV)
print("Segmentos regulares:", DIARIZATION_ALL_REGULAR_SEGMENTS_CSV)
print("Segmentos puntuados:", DIARIZATION_ALL_SCORED_SEGMENTS_CSV)
print("Segmentos válidos:", DIARIZATION_ALL_VALID_SEGMENTS_CSV)
print("Anchors:", DIARIZATION_ALL_ANCHOR_SEGMENTS_CSV)

df_summary

[1200/1200] Saltando ya procesado: raw_bajas_9157454659260016851_clean.wav
Procesados nuevos: 0 | Reusados: 1198 | Errores: 2

Diarización finalizada o reanudada correctamente.
Audios nuevos procesados en esta ejecución: 0
Audios reutilizados desde outputs previos: 1198
Errores: 2
Resumen: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/diarization_summary.csv
Segmentos regulares: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/diarization_all_regular_segments.csv
Segmentos puntuados: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/diarization_all_scored_segments.csv
Segmentos válidos: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/diarization_all_valid_segments.csv
Anchors: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/diarization_all_anchor_segments.csv


,audio_file,audio_stem,sample_rate,duration_sec,diarization_mode,n_regular_segments,n_scored_segments,n_valid_segments,n_anchor_segments,n_speakers,n_overlap_regions,regular_csv_path,raw_csv_path,clean_csv_path,anchors_csv_path,rttm_path
0,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,NaN,171.362844,from_checkpoint,84,67,42,6,2,28,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
1,raw_9154117551220006851_clean.wav,raw_9154117551220006851_clean,NaN,175.581594,from_checkpoint,79,72,41,6,2,12,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
2,raw_9154127337680006851_clean.wav,raw_9154127337680006851_clean,NaN,143.519094,from_checkpoint,87,69,36,6,2,26,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
3,raw_9154142438160016851_clean.wav,raw_9154142438160016851_clean,NaN,265.457844,from_checkpoint,114,79,50,6,2,16,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
4,raw_9154152155960016851_clean.wav,raw_9154152155960016851_clean,NaN,172.341594,from_checkpoint,103,88,42,6,2,18,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1193,raw_bajas_9157452496450006851_clean.wav,raw_bajas_9157452496450006851_clean,NaN,175.682844,from_checkpoint,39,21,16,6,2,6,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
1194,raw_bajas_9157452583930006851_clean.wav,raw_bajas_9157452583930006851_clean,NaN,268.731594,from_checkpoint,122,97,64,6,2,38,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
1195,raw_bajas_9157452646670006851_clean.wav,raw_bajas_9157452646670006851_clean,NaN,83.882844,from_checkpoint,30,21,17,6,2,3,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...
1196,raw_bajas_9157453715050006851_clean.wav,raw_bajas_9157453715050006851_clean,NaN,160.967844,from_checkpoint,69,52,32,6,2,5,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...,/home/jupyter/TFM_ProcesadoDeAudios/data/diari...


In [9]:
# VISTA RAPIDA DEL CONSOLIDADO

print("Total de audios procesados:", len(df_summary))
print("Total de segmentos puntuados:", len(df_all_scored_segments))
print("Total de segmentos válidos:", len(df_all_valid_segments))
print("Total de anchors:", len(df_all_anchor_segments))

df_summary[[
    "audio_file", "duration_sec", "n_regular_segments",
    "n_scored_segments", "n_valid_segments",
    "n_anchor_segments", "n_overlap_regions", "n_speakers"
]]

Total de audios procesados: 1198
Total de segmentos puntuados: 73000
Total de segmentos válidos: 43867
Total de anchors: 7038


,audio_file,duration_sec,n_regular_segments,n_scored_segments,n_valid_segments,n_anchor_segments,n_overlap_regions,n_speakers
0,raw_9154117451310006851_clean.wav,171.362844,84,67,42,6,28,2
1,raw_9154117551220006851_clean.wav,175.581594,79,72,41,6,12,2
2,raw_9154127337680006851_clean.wav,143.519094,87,69,36,6,26,2
3,raw_9154142438160016851_clean.wav,265.457844,114,79,50,6,16,2
4,raw_9154152155960016851_clean.wav,172.341594,103,88,42,6,18,2
...,...,...,...,...,...,...,...,...
1193,raw_bajas_9157452496450006851_clean.wav,175.682844,39,21,16,6,6,2
1194,raw_bajas_9157452583930006851_clean.wav,268.731594,122,97,64,6,38,2
1195,raw_bajas_9157452646670006851_clean.wav,83.882844,30,21,17,6,3,2
1196,raw_bajas_9157453715050006851_clean.wav,160.967844,69,52,32,6,5,2


## FASE 01.2 - MODELO DE EMBEDDING

In [10]:
# RESTAURAR OUTPUTS DE RELABELING DESDE GCS

if not RE_RELABEL:
    download_directory(
        local_dir=FINAL_RELABEL_DIR,
        gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
        gcs_client=gcs_client,
        base_dir=OUTPUT_DIR,
        clear_output_fn=clear_output,
    )
else:
    print(
        "Restauración de relabeling omitida "
        "porque RE_RELABEL=True."
    )

Restauración desde GCS completada.
Archivos encontrados: 7111
Archivos descargados: 0
Archivos locales ya vigentes: 7111


In [11]:

# MODELO DE EMBEDDINGS

ensure_relabel_directories()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_model = PretrainedSpeakerEmbedding(
    EMBEDDING_MODEL_ID,
    device=device,
    token=HF_TOKEN,
)

print("Embedding model cargado:", EMBEDDING_MODEL_ID)
print("Device:", device)
print("Sample rate esperado:", EMBEDDING_SAMPLE_RATE)
print("RELABEL_MIN_MARGIN:", RELABEL_MIN_MARGIN)

Embedding model cargado: pyannote/wespeaker-voxceleb-resnet34-LM
Device: cpu
Sample rate esperado: 16000
RELABEL_MIN_MARGIN: 0.01


In [12]:
# RELABELING FINAL Y CONSOLIDACIÓN

relabel_outputs = run_relabeling_pipeline(
    wav_files=wav_files,
    embedding_model=embedding_model,
    max_gap_merge_sec=MAX_GAP_MERGE_SEC,
    force_relabel=RE_RELABEL,
)

df_relabel_summary = relabel_outputs["df_relabel_summary"]
df_all_final_segments = relabel_outputs["df_all_final_segments"]
df_all_final_merged = relabel_outputs["df_all_final_merged"]
df_all_anchor_embeddings = relabel_outputs["df_all_anchor_embeddings"]
df_all_changed_segments = relabel_outputs["df_all_changed_segments"]
df_all_anchor_embedding_vectors = relabel_outputs["df_all_anchor_embedding_vectors"]
df_all_segment_embedding_vectors = relabel_outputs["df_all_segment_embedding_vectors"]

Outputs consolidados de reetiquetado encontrados. Se reutilizan sin volver a procesar los audios.


In [13]:
# Resumen rápido

df_relabel_summary[[
    "audio_file",
    "n_valid_in",
    "n_anchor_in",
    "n_anchor_embeddings",
    "n_anchor_speakers",
    "n_changed_segments",
    "n_final_segments",
    "n_final_merged_segments",
    "mean_distance_margin",
    "status"
]]


,audio_file,n_valid_in,n_anchor_in,n_anchor_embeddings,n_anchor_speakers,n_changed_segments,n_final_segments,n_final_merged_segments,mean_distance_margin,status
0,raw_9154117451310006851_clean.wav,42,6,6,2,1,42,37.0,0.426726,ok
1,raw_9154117551220006851_clean.wav,41,6,6,2,0,41,41.0,0.395578,ok
2,raw_9154127337680006851_clean.wav,36,6,6,2,7,36,27.0,0.269856,ok
3,raw_9154142438160016851_clean.wav,50,6,6,2,3,50,46.0,0.460637,ok
4,raw_9154152155960016851_clean.wav,42,6,6,2,3,42,38.0,0.349696,ok
...,...,...,...,...,...,...,...,...,...,...
1195,raw_bajas_9157452496450006851_clean.wav,16,6,6,2,1,16,16.0,0.237892,ok
1196,raw_bajas_9157452583930006851_clean.wav,64,6,6,2,2,64,56.0,0.503840,ok
1197,raw_bajas_9157452646670006851_clean.wav,17,6,6,2,3,17,17.0,0.173947,ok
1198,raw_bajas_9157453715050006851_clean.wav,32,6,6,2,5,32,30.0,0.090324,ok


In [14]:
# ============================================================
# RESUMEN GENERAL DEL RELABELING + EJEMPLO CON CAMBIOS
# ============================================================

# Buscar únicamente archivos individuales por audio
final_segments_files = sorted(
    path for path in FINAL_RELABEL_DIR.glob("*_final_segments.csv")
    if not path.name.startswith("all_")
)

final_merged_files = sorted(
    path for path in FINAL_RELABEL_DIR.glob("*_final_merged.csv")
    if not path.name.startswith("all_")
)

changed_segments_files = sorted(
    path for path in FINAL_RELABEL_DIR.glob("*_changed_segments.csv")
    if not path.name.startswith("all_")
)

print("Archivos final_segments encontrados:", len(final_segments_files))
print("Archivos final_merged encontrados:", len(final_merged_files))
print("Archivos changed_segments encontrados:", len(changed_segments_files))

if not final_segments_files:
    raise FileNotFoundError(
        "Todavía no hay archivos *_final_segments.csv individuales en final_relabel/."
    )


# ------------------------------------------------------------
# 1. Resumen general del relabeling por audio
# ------------------------------------------------------------

summary_rows = []

for final_path in final_segments_files:
    audio_base = final_path.name.replace("_final_segments.csv", "")

    try:
        df_audio = pd.read_csv(final_path)
    except Exception as error:
        print("No se pudo leer:", final_path.name, "|", error)
        continue

    n_segments = len(df_audio)

    if n_segments == 0:
        continue

    if "was_reclassified" in df_audio.columns:
        n_relabelled = int(
            df_audio["was_reclassified"]
            .fillna(False)
            .astype(bool)
            .sum()
        )
    elif {"speaker_original", "speaker_final"}.issubset(df_audio.columns):
        n_relabelled = int(
            (
                df_audio["speaker_original"]
                != df_audio["speaker_final"]
            ).sum()
        )
    else:
        n_relabelled = 0

    summary_rows.append({
        "audio_base": audio_base,
        "audio_file": f"{audio_base}.wav",
        "n_segments": n_segments,
        "n_relabelled": n_relabelled,
        "relabel_ratio": n_relabelled / n_segments,
        "mean_distance_margin": (
            df_audio["distance_margin"].mean()
            if "distance_margin" in df_audio.columns
            else np.nan
        ),
        "median_distance_margin": (
            df_audio["distance_margin"].median()
            if "distance_margin" in df_audio.columns
            else np.nan
        ),
        "max_distance_margin": (
            df_audio["distance_margin"].max()
            if "distance_margin" in df_audio.columns
            else np.nan
        ),
    })

df_relabeling_summary_by_audio = pd.DataFrame(summary_rows)

if df_relabeling_summary_by_audio.empty:
    raise ValueError("No se pudo construir el resumen de relabeling.")

total_audios = df_relabeling_summary_by_audio["audio_base"].nunique()
total_segments = int(df_relabeling_summary_by_audio["n_segments"].sum())
total_relabelled = int(df_relabeling_summary_by_audio["n_relabelled"].sum())

global_relabel_ratio = (
    total_relabelled / total_segments
    if total_segments > 0
    else np.nan
)

audios_with_relabeling = int(
    (df_relabeling_summary_by_audio["n_relabelled"] > 0).sum()
)

audios_without_relabeling = total_audios - audios_with_relabeling

print("\n==============================")
print("RESUMEN GENERAL DEL RELABELING")
print("==============================")
print("Audios analizados:", total_audios)
print("Segmentos finales analizados:", total_segments)
print("Segmentos reetiquetados:", total_relabelled)
print("Porcentaje global reetiquetado:", round(global_relabel_ratio * 100, 2), "%")
print("Audios con al menos un segmento reetiquetado:", audios_with_relabeling)
print("Audios sin segmentos reetiquetados:", audios_without_relabeling)

write_csv_atomic(
    df_relabeling_summary_by_audio,
    RELABELING_SUMMARY_BY_AUDIO_CSV,
)

print("\nResumen por audio guardado en:")
print(RELABELING_SUMMARY_BY_AUDIO_CSV)

display(
    df_relabeling_summary_by_audio
    .sort_values("n_relabelled", ascending=False)
    .head(15)
)


# ------------------------------------------------------------
# 2. Seleccionar automáticamente un audio con cambios
# ------------------------------------------------------------

df_candidates = (
    df_relabeling_summary_by_audio
    .query("n_relabelled > 0")
    .sort_values(
        ["n_relabelled", "relabel_ratio"],
        ascending=False,
    )
    .reset_index(drop=True)
)

if df_candidates.empty:
    raise ValueError(
        "No hay audios con segmentos reetiquetados entre los archivos disponibles."
    )


df_candidates["files_exist"] = (
    df_candidates["audio_base"]
    .apply(validation_inputs_exist)
)

df_candidates = (
    df_candidates[df_candidates["files_exist"]]
    .copy()
    .reset_index(drop=True)
)

if df_candidates.empty:
    raise ValueError(
        "Hay audios con relabeling, pero ninguno tiene todos los archivos "
        "necesarios para construir la tabla maestra."
    )

audio_to_validate = df_candidates.loc[0, "audio_base"]

print("\n==============================")
print("AUDIO SELECCIONADO CON RELABELING")
print("==============================")
print("Audio:", audio_to_validate)
print("Segmentos reetiquetados:", int(df_candidates.loc[0, "n_relabelled"]))
print(
    "Ratio reetiquetado:",
    round(df_candidates.loc[0, "relabel_ratio"] * 100, 2),
    "%",
)


# ------------------------------------------------------------
# 3. Construir tabla maestra del audio seleccionado
# ------------------------------------------------------------

df_validation = build_validation_table(audio_to_validate)

cols_to_show = [
    "segment_id_raw",
    "start_raw",
    "end_raw",
    "duration_raw",
    "speaker_raw",
    "speaker_original",
    "speaker_final",
    "was_reclassified",
    "distance_margin",
    "is_anchor",
    "anchor_rank",
    "overlap_ratio_raw",
    "rms_dbfs_raw",
    "valid_export_raw",
    "merge_start",
    "merge_end",
    "merge_speaker_final",
]

cols_to_show = [
    column
    for column in cols_to_show
    if column in df_validation.columns
]

print("\nTabla de validación construida correctamente.")
print("Filas:", len(df_validation))
print("Columnas:", len(df_validation.columns))

if "was_reclassified" in df_validation.columns:
    df_changed_view = df_validation[
        df_validation["was_reclassified"]
        .fillna(False)
        .astype(bool)
    ].copy()
else:
    df_changed_view = pd.DataFrame()

print("\nSegmentos reetiquetados en este audio:", len(df_changed_view))

if not df_changed_view.empty:
    display(df_changed_view[cols_to_show].head(30))
else:
    display(df_validation[cols_to_show].head(50))

Archivos final_segments encontrados: 1181
Archivos final_merged encontrados: 1181
Archivos changed_segments encontrados: 1181

RESUMEN GENERAL DEL RELABELING
Audios analizados: 1181
Segmentos finales analizados: 43493
Segmentos reetiquetados: 3700
Porcentaje global reetiquetado: 8.51 %
Audios con al menos un segmento reetiquetado: 933
Audios sin segmentos reetiquetados: 248

Resumen por audio guardado en:
/home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/final_relabel/relabeling_summary_by_audio.csv


,audio_base,audio_file,n_segments,n_relabelled,relabel_ratio,mean_distance_margin,median_distance_margin,max_distance_margin
742,raw_bajas_9156993504100006851_clean,raw_bajas_9156993504100006851_clean.wav,72,29,0.402778,0.101392,0.107203,0.229827
712,raw_bajas_9156942885910006851_clean,raw_bajas_9156942885910006851_clean.wav,43,21,0.488372,0.198602,0.179826,0.453174
595,raw_bajas_9156702040930006851_clean,raw_bajas_9156702040930006851_clean.wav,48,20,0.416667,0.306338,0.307691,0.517978
189,raw_9156845884280006851_clean,raw_9156845884280006851_clean.wav,46,19,0.413043,0.126758,0.104258,0.329930
24,raw_9154394927400006851_clean,raw_9154394927400006851_clean.wav,72,19,0.263889,0.099602,0.091815,0.234024
210,raw_bajas_9156002528450026851_clean,raw_bajas_9156002528450026851_clean.wav,67,18,0.268657,0.263237,0.249661,0.655054
309,raw_bajas_9156258566490016851_clean,raw_bajas_9156258566490016851_clean.wav,43,18,0.418605,0.076707,0.069167,0.207096
1027,raw_bajas_9157339371880006851_clean,raw_bajas_9157339371880006851_clean.wav,33,17,0.515152,0.099664,0.101173,0.189625
621,raw_bajas_9156762393720006851_clean,raw_bajas_9156762393720006851_clean.wav,67,17,0.253731,0.264040,0.278654,0.537877
375,raw_bajas_9156362641780006851_clean,raw_bajas_9156362641780006851_clean.wav,45,16,0.355556,0.115722,0.097559,0.332058



AUDIO SELECCIONADO CON RELABELING
Audio: raw_bajas_9156993504100006851_clean
Segmentos reetiquetados: 29
Ratio reetiquetado: 40.28 %

Tabla de validación construida correctamente.
Filas: 168
Columnas: 34

Segmentos reetiquetados en este audio: 29


/home/jupyter/TFM_ProcesadoDeAudios/src/reetiquetado_embeddings.py:1121: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_valid_segment"].fillna(False)
/home/jupyter/TFM_ProcesadoDeAudios/src/reetiquetado_embeddings.py:1148: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["is_anchor"] = df["is_anchor"].fillna(False)
/home/jupyter/TFM_ProcesadoDeAudios/src/reetiquetado_embeddings.py:1196: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) i

,segment_id_raw,start_raw,end_raw,duration_raw,speaker_raw,speaker_original,speaker_final,was_reclassified,distance_margin,is_anchor,anchor_rank,overlap_ratio_raw,rms_dbfs_raw,valid_export_raw,merge_start,merge_end,merge_speaker_final
1,2,1.549719,2.275344,0.725625,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.070518,False,NaN,0.000000,-20.364464,True,1.549719,2.275344,SPEAKER_01
12,13,11.033469,11.978469,0.945000,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.057591,False,NaN,0.000000,-16.691641,True,11.033469,15.184719,SPEAKER_01
14,15,12.029094,12.974094,0.945000,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.148887,False,NaN,0.000000,-17.953585,True,11.033469,15.184719,SPEAKER_01
16,17,13.024719,13.969719,0.945000,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.087801,False,NaN,0.000000,-18.483986,True,11.033469,15.184719,SPEAKER_01
18,19,14.037219,15.184719,1.147500,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.080748,False,NaN,0.000000,-17.405050,True,11.033469,15.184719,SPEAKER_01
26,27,30.034719,31.300344,1.265625,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.071245,False,NaN,0.000000,-19.902664,True,23.470344,31.300344,SPEAKER_01
27,28,31.958469,33.004719,1.046250,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.047153,False,NaN,0.500000,-17.978657,True,31.958469,33.004719,SPEAKER_01
29,30,35.789094,37.611594,1.822500,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.105993,False,NaN,0.000000,-16.833836,True,33.662844,37.611594,SPEAKER_01
43,44,52.697844,55.380969,2.683125,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.045576,False,NaN,0.144654,-14.582354,True,52.697844,58.806594,SPEAKER_01
64,65,88.675344,89.400969,0.725625,SPEAKER_00,SPEAKER_00,SPEAKER_01,True,0.114599,False,NaN,0.953488,-28.907316,True,80.862219,89.400969,SPEAKER_01


In [15]:
# SUBIDA FINAL DE OUTPUTS NUEVOS O MODIFICADOS A GCS

upload_directory(
    local_dir=OUTPUT_DIR,
    gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
    gcs_client=gcs_client,
    clear_output_fn=clear_output,
    skip_unchanged=True,
)

Subida final completada.
Archivos locales revisados: 13376
Archivos subidos: 220
Archivos omitidos sin cambios: 13156
Destino: gs://catedras_audio_detection/pipelineA/procesados_UNAV/diarization_outputs/


{'total': 13376, 'uploaded': 220, 'skipped': 13156}

## Estructura final esperada

- `outputs/<nombre_audio>_raw.csv` → segmentos usados para puntuar/limpiar
- `outputs/<nombre_audio>.csv` → segmentos válidos para exportar y escuchar
- `outputs/<nombre_audio>_anchors.csv` → anchors de alta confianza por speaker
- `outputs/<nombre_audio>.rttm`
- `outputs/diarization_summary.csv`
- `outputs/diarization_all_regular_segments.csv`
- `outputs/diarization_all_scored_segments.csv`
- `outputs/diarization_all_valid_segments.csv`
- `outputs/diarization_all_anchor_segments.csv`
- `outputs/segments/<nombre_audio>/...wav`
- `outputs/anchor_segments/<nombre_audio>/...wav`

### Idea de uso
- Los **segmentos válidos** sirven para auditoría general.
- Los **anchors** sirven para la siguiente fase de identificación de speaker/rol.
- El **inicio superpuesto** no se elimina del audio, pero sí se puede excluir del proceso de seleccionar anchors.
